# Phase 1 — Patient-Level 5-Fold Cross-Validation
**Goal:** Establish reliable, confidence-interval-bounded baseline metrics for Model A
**Why now:** Our test set has only 14 R2 and 9 R3A patients. A single mis-prediction swings R2 sensitivity by 7 pp. We need variance estimates, not point estimates.
**Strategy:** Train 5 linear-probe models (frozen backbone), each on 4/5 of patients, evaluate on the 1/5 held out. Pool all predictions → bootstrap CIs.
**Fixed test set:** The 175-patient test set is never touched during CV — it is used only for the final ensemble evaluation at the end.


In [1]:
import os, sys, json, copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate truncated JPEGs

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

# ── Working directory: project root ───────────────────────────────────────────
os.chdir(os.path.dirname(os.path.abspath('phase1_cross_validation.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Config ────────────────────────────────────────────────────────────────────
N_FOLDS       = 5
MAX_EPOCHS    = 50
PATIENCE      = 10      # early stopping patience on fold-val AUROC
BATCH_SIZE    = 64      # same as LP training
INPUT_SIZE    = 224
LR            = 1.25e-3 # blr=5e-3 * batch_size(64) / 256  (matches LP)
MIN_LR        = 1e-6
WARMUP_EPOCHS = 5
WEIGHT_DECAY  = 0.05
NUM_CLASSES   = 4
CLASS_WEIGHTS = [1.0, 1.7851, 9.5294, 15.6774]  # same as LP/FT training
SEED          = 42

# Pretrained backbone (same HuggingFace repo used for LP/FT)
HF_REPO       = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE       = 'RETFound_dinov2_meh.pth'

# Output directory for CV results
CV_OUTPUT     = Path('output_dir/phase1_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Config: {N_FOLDS}-fold CV, {MAX_EPOCHS} max epochs, patience={PATIENCE}')


Working directory: /home/eth/Desktop/Isaack/RETFound-main
Device: cuda
Config: 5-fold CV, 50 max epochs, patience=10


## Why do we need cross-validation? The instability problem

Between your two LP runs (identical config, different random seeds), R2 sensitivity swung from 0.549 to 0.667, and R3A from 0.800 to 0.600. That is not model instability — it is **test-set size instability**.

| Metric | LP Run 1 | LP Run 2 | Δ |
|--------|----------|----------|---|
| AUROC  | 0.841    | 0.843    | +0.002 |
| R2 sensitivity | 0.549 | 0.667 | **+0.118** |
| R3A sensitivity | 0.800 | 0.600 | **-0.200** |

With 12 R2 patients in the test set, one patient's prediction changes sensitivity by 1/12 ≈ 8 pp.
With 9 R3A patients, one change = 11 pp.

**Cross-validation pools predictions from 990 patients** (all train+val) instead of 175, giving a denominator ~6× larger. This alone cuts the per-patient variance by √6 ≈ 2.4×.
Bootstrap CIs then let us say: "R2 sensitivity is 0.58 ± 0.09" rather than a point estimate we can't trust.


In [2]:
# ── Load splits.csv ───────────────────────────────────────────────────────────
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}
CLASSES = ['R0', 'R1', 'R2', 'R3A']

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

# Separate CV data (train+val) and the sacred test set
df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

# Patient-level worst grade (used for stratification only)
# A patient with LE=R0 and RE=R2 is treated as R2 for stratification so rare
# classes are distributed evenly across folds.
pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']

patient_ids  = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

print(f'CV pool   : {len(df_cv):>5} images  |  {len(patient_ids):>4} patients')
print(f'Test set  : {len(df_test):>5} images  |  {df_test["code"].nunique():>4} patients')
print()
print('CV patient-level class distribution (worst-eye grade):')
for i, c in enumerate(CLASSES):
    n = (patient_strat == i).sum()
    print(f'  {c}: {n:3d} patients  ({n/len(patient_strat)*100:.1f}%)')
print()
print('Note: 147 patients have mixed grades between eyes.')
print('Each image is still labelled by ITS OWN EYE grade during training.')


CV pool   :  4075 images  |   990 patients
Test set  :   702 images  |   175 patients

CV patient-level class distribution (worst-eye grade):
  R0: 518 patients  (52.3%)
  R1: 362 patients  (36.6%)
  R2:  67 patients  (6.8%)
  R3A:  43 patients  (4.3%)

Note: 147 patients have mixed grades between eyes.
Each image is still labelled by ITS OWN EYE grade during training.


## Patient-level splitting: preventing data leakage

Patient `0001_T` has 4 images: LE×2 and RE×2. If we split images randomly, the model might train on LE images and be evaluated on RE images from the same person. Because both retinas share the same patient's vascular architecture, the model can "recognise the patient" rather than the disease — artificially inflating metrics.

**Rule: all images from the same patient must land in the same fold.**

We assign folds at the patient level, then collect every image that belongs to each patient-fold assignment.

`StratifiedKFold` divides patients into 5 equal groups while preserving the class ratio of each fold. The stratification key is each patient's **worst-eye grade** — this ensures rare-grade patients (R2, R3A) are distributed as evenly as possible across folds rather than accidentally clumping in one fold.


In [3]:
# ── Assign each patient to a fold ─────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# fold_assignments[patient_code] = fold_index (0..4)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx

pat_grade['fold'] = pat_grade['code'].map(fold_assignments)

print(f'Fold assignments (patient counts):')
print()
header = f"{'Grade':<6}" + "".join(f"  Fold {i}" for i in range(N_FOLDS)) + "  Total"
print(header)
print('-' * len(header))
for g, name in enumerate(CLASSES):
    row = pat_grade[pat_grade['strat_grade'] == g]
    counts = [len(row[row['fold'] == f]) for f in range(N_FOLDS)]
    print(f'{name:<6}' + "".join(f"  {c:5d}" for c in counts) + f"  {sum(counts):5d}")
total_row = [len(pat_grade[pat_grade['fold'] == f]) for f in range(N_FOLDS)]
print('-' * len(header))
print(f"{'Total':<6}" + "".join(f"  {c:5d}" for c in total_row) + f"  {len(pat_grade):5d}")
print()
print('Train images per fold (4 folds combined):')
for f in range(N_FOLDS):
    train_pats = pat_grade[pat_grade['fold'] != f]['code'].values
    val_pats   = pat_grade[pat_grade['fold'] == f]['code'].values
    n_train    = len(df_cv[df_cv['code'].isin(train_pats)])
    n_val      = len(df_cv[df_cv['code'].isin(val_pats)])
    print(f'  Fold {f}: {n_train} train images, {n_val} val images')


Fold assignments (patient counts):

Grade   Fold 0  Fold 1  Fold 2  Fold 3  Fold 4  Total
-----------------------------------------------------
R0        104    104    104    103    103    518
R1         72     72     72     73     73    362
R2         14     14     13     13     13     67
R3A         8      8      9      9      9     43
-----------------------------------------------------
Total     198    198    198    198    198    990

Train images per fold (4 folds combined):
  Fold 0: 3257 train images, 818 val images
  Fold 1: 3242 train images, 833 val images
  Fold 2: 3256 train images, 819 val images
  Fold 3: 3271 train images, 804 val images
  Fold 4: 3274 train images, 801 val images


## Dataset class: reading images from a path list

The existing `build_dataset` in `util/datasets.py` reads from a directory tree (`image_trees/modelA/train/R0/...`). For CV we need to build datasets from arbitrary lists of `(image_path, label)` pairs — one list per fold.

`RetinopathyDataset` wraps any such list and applies the same transforms used during the original LP training:
- **Train transform**: RandAugment (`rand-m9-mstd0.5-inc1`), RandomErasing (p=0.25), random flip/crop, ImageNet normalisation
- **Val/test transform**: resize → centre-crop → normalise (no augmentation)

These are exactly the same settings from `util/datasets.build_transform`.


In [4]:
import argparse
from util.datasets import build_transform  # reuse existing transform builder

# Build a minimal args-like object for build_transform
_aug_args = argparse.Namespace(
    input_size   = INPUT_SIZE,
    color_jitter = None,
    aa           = 'rand-m9-mstd0.5-inc1',
    reprob       = 0.25,
    remode       = 'pixel',
    recount      = 1,
)
train_transform = build_transform('train', _aug_args)
eval_transform  = build_transform('val',   _aug_args)

class RetinopathyDataset(Dataset):
    """Dataset backed by a list of (image_path, label_int) tuples."""
    def __init__(self, records, transform):
        self.records   = records    # [(path_str, label_int), ...]
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label = self.records[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


def make_records(df_subset):
    """Convert a DataFrame slice into (image_path, label) records."""
    return [(row.image_path, row.grade_int) for row in df_subset.itertuples()]


# Quick sanity check
sample_recs = make_records(df_cv.head(4))
ds = RetinopathyDataset(sample_recs, eval_transform)
img, lbl = ds[0]
print(f'Sample image tensor shape: {img.shape}  label: {lbl}  ({CLASSES[lbl]})')


/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample image tensor shape: torch.Size([3, 224, 224])  label: 0  (R0)


## Loading the pretrained backbone

We use the same weights as the original LP training:
`YukunZhou/RETFound_dinov2_meh` from Hugging Face (already cached from the earlier runs).

`load_backbone()` does four things:
1. **Build** the ViT-Large architecture via timm (same call as `models_vit.RETFound_dinov2`)
2. **Download** the RETFound checkpoint (cached after first download, typically ~1.2 GB)
3. **Load** the teacher weights, applying the same key-name fixes as `main_finetune.py`
4. **Freeze** all parameters except `head.weight` and `head.bias` (1024×4 + 4 = 4100 params)
5. **Re-initialise** the head with a fresh tiny normal (std=2e-5) for each fold

This means every fold starts from the same frozen feature extractor with a fresh linear head — exactly what LP training does.


In [ ]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone(device, num_classes=NUM_CLASSES, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    # Build ViT-Large — timm loads DINOv2 weights with pos_embed already at 224px
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True,
        img_size=INPUT_SIZE,
        num_classes=num_classes,
        drop_path_rate=0.2,
    )

    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}

    # Remove any key whose shape doesn't match the model (pos_embed is 518px in
    # the checkpoint but 224px in the model; head keys are 1000-class vs 4-class).
    # timm already loaded the correct 224px pos_embed from DINOv2 weights above.
    state = model.state_dict()
    drop = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        print(f'  Skipping mismatched key: {k}  {tuple(ckpt_model[k].shape)} -> keeping model init')
        del ckpt_model[k]

    model.load_state_dict(ckpt_model, strict=False)

    # Fresh head
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)

    # Freeze everything except the head
    for name, param in model.named_parameters():
        param.requires_grad = ('head' in name)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)')
    return model.to(device)


print('Loading backbone to verify...')
_test_model = load_backbone(device=DEVICE)
print('Backbone loaded OK.')
del _test_model
torch.cuda.empty_cache()

## Early stopping and the training helpers

**Why early stopping?** The existing LP ran for all 50 epochs and selected the best checkpoint by val AUROC after the fact. For CV we implement true early stopping: if val AUROC does not improve for `PATIENCE=10` epochs, training stops and the best weights are restored. This saves roughly 10–20 epochs per fold.

`EarlyStopping` stores only the **head weights** (8 KB), not the full model (1.2 GB), since the backbone is frozen and never changes.

**`train_epoch`** mirrors `engine_finetune.train_one_epoch`:
- Mixed precision (AMP)
- Cosine LR schedule with linear warmup
- Weighted CrossEntropyLoss (same class weights as LP/FT)

**`eval_fold`** collects softmax probabilities on the held-out fold for metric computation.

**`compute_metrics`** computes AUROC, accuracy, κ, macro sensitivity, macro specificity, and per-class sensitivity/specificity — the same metrics as `engine_finetune.evaluate`.


In [ ]:
import math

# ── Early stopping (stores head weights only) ──────────────────────────────────
class EarlyStopping:
    def __init__(self, patience, model):
        self.patience   = patience
        self.best_auroc = -float('inf')
        self.counter    = 0
        # Seed best_head with initial weights so restore() always has valid weights,
        # even if step() never records an improvement (e.g. all-NaN AUROCs).
        self.best_head  = copy.deepcopy(model.head.state_dict())

    def step(self, auroc, model):
        # NaN guard: nan > -inf is False in Python, which silently wastes patience
        # without ever saving a checkpoint — treat NaN as 0.0 instead.
        if auroc != auroc:
            auroc = 0.0
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            self.best_head  = copy.deepcopy(model.head.state_dict())
            return False            # do not stop
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        model.head.load_state_dict(self.best_head)


# ── LR schedule (warmup + cosine, same as util/lr_sched.py) ──────────────────
def get_lr(epoch, warmup_epochs, max_epochs, lr, min_lr):
    if epoch < warmup_epochs:
        return lr * (epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * t))


# ── Train one epoch ────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device, scaler, epoch,
                warmup_epochs, max_epochs, lr, min_lr):
    model.train()
    cur_lr = get_lr(epoch, warmup_epochs, max_epochs, lr, min_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = cur_lr

    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)

    return total_loss / len(loader.dataset), cur_lr


# ── Evaluate: return (labels_np, probs_np) ────────────────────────────────────
@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().float()
        all_labels.append(labels)
        all_probs.append(probs)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()


# ── Compute classification metrics ────────────────────────────────────────────
def compute_metrics(labels, probs):
    preds = probs.argmax(axis=1)
    # Cast to float64 and renormalize: float32 softmax rows sum to ~0.9999,
    # which fails sklearn's strict sum-to-1 check in roc_auc_score.
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro')
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')

    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))

    return {
        'auroc':              auroc,
        'accuracy':           acc,
        'kappa':              kappa,
        'macro_sensitivity':  np.nanmean(sens),
        'macro_specificity':  np.nanmean(spec),
        'sensitivity':        np.array(sens),
        'specificity':        np.array(spec),
    }


print('Helpers defined.')

## Running 5-fold cross-validation

The cell below trains all 5 folds. For each fold:

1. **Split images** into fold-train (4 folds' images) and fold-val (1 fold's images)
2. **Load a fresh backbone** (same pretrained weights, fresh head)
3. **Train** with early stopping (patience=10 on fold-val AUROC)
4. **Restore best head**, then collect:
   - OOF (out-of-fold) predictions on fold-val patients
   - Predictions on the fixed test set (for ensemble later)
5. **Save** predictions to disk so the analysis cells can be re-run without retraining

**Expected runtime:** ~30–60 minutes per fold on GPU (LP training is fast because only the 4,100-parameter head backprops; the backbone is frozen).
Each fold typically stops around epoch 20–30 due to early stopping.

Progress is printed every epoch.


In [ ]:
weight_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float).to(DEVICE)
criterion_cv  = nn.CrossEntropyLoss(weight=weight_tensor)

# Storage for pooled OOF predictions (filled fold by fold)
oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)

# Map each CV image row to its position in the pooled array
df_cv = df_cv.reset_index(drop=True)
df_cv['cv_idx'] = df_cv.index  # 0 … len(df_cv)-1

fold_results = []   # per-fold summary for quick inspection

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*60}')

    # ── Split images by patient fold assignment ────────────────────────────────
    val_pats   = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats = pat_grade[pat_grade['fold'] != fold]['code'].values

    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]

    val_cv_indices = df_fold_val['cv_idx'].values   # positions in pooled array

    print(f'  Train: {len(df_fold_train)} images | Val: {len(df_fold_val)} images')

    train_recs = make_records(df_fold_train)
    val_recs   = make_records(df_fold_val)
    test_recs  = make_records(df_test)

    ds_train = RetinopathyDataset(train_recs, train_transform)
    ds_val   = RetinopathyDataset(val_recs,   eval_transform)
    ds_test  = RetinopathyDataset(test_recs,  eval_transform)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=False)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    # ── Fresh model + optimizer for this fold ─────────────────────────────────
    model     = load_backbone(device=DEVICE, seed=SEED + fold)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    scaler  = torch.cuda.amp.GradScaler()
    stopper = EarlyStopping(patience=PATIENCE, model=model)

    # ── Training loop ─────────────────────────────────────────────────────────
    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch(
            model, loader_train, optimizer, criterion_cv, DEVICE, scaler,
            epoch, WARMUP_EPOCHS, MAX_EPOCHS, LR, MIN_LR,
        )
        val_labels, val_probs = eval_fold(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)

        elapsed = time.time() - t_start
        print(f'  ep {epoch:02d} | lr={cur_lr:.2e} | loss={tr_loss:.4f} | '
              f'AUROC={m["auroc"]:.4f} | sens={m["macro_sensitivity"]:.4f} | '
              f'elapsed={elapsed:.0f}s')

        if stopper.step(m['auroc'], model):
            print(f'  Early stop at epoch {epoch} (best AUROC={stopper.best_auroc:.4f})')
            break

    # ── Restore best head ─────────────────────────────────────────────────────
    stopper.restore(model)
    print(f'  Best val AUROC: {stopper.best_auroc:.4f}')

    # ── Collect OOF predictions ───────────────────────────────────────────────
    oof_labels, oof_probs = eval_fold(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs

    # ── Test set predictions from this fold ───────────────────────────────────
    test_labels_fold, test_probs_fold = eval_fold(model, loader_test, DEVICE)

    # ── Save fold predictions to disk ─────────────────────────────────────────
    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',  oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy', oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy', test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy', test_labels_fold)

    m_fold = compute_metrics(oof_labels, oof_probs)
    fold_results.append({
        'fold':              fold,
        'best_auroc':        stopper.best_auroc,
        'oof_auroc':         m_fold['auroc'],
        'oof_accuracy':      m_fold['accuracy'],
        'oof_kappa':         m_fold['kappa'],
        'oof_macro_sens':    m_fold['macro_sensitivity'],
        'oof_macro_spec':    m_fold['macro_specificity'],
    })
    print(f'  OOF AUROC: {m_fold["auroc"]:.4f}  Kappa: {m_fold["kappa"]:.4f}')

    del model
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('All folds complete.')
np.save(CV_OUTPUT / 'oof_labels_all.npy', oof_labels_all)
np.save(CV_OUTPUT / 'oof_probs_all.npy',  oof_probs_all)
with open(CV_OUTPUT / 'fold_results.json', 'w') as f:
    json.dump(fold_results, f, indent=2)
print(f'Saved pooled OOF predictions to {CV_OUTPUT}/')

## Bootstrap confidence intervals: from point estimates to error bars

**What bootstrap does:**
We have 990 OOF predictions (each patient predicted exactly once by the fold where they were held out). We resample this pool 1000 times **with replacement** — each resample is 990 random draws from the pool. We compute a metric on each resample and record it. The 2.5th and 97.5th percentiles of those 1000 values form a 95% CI.

**Why "with replacement"?**
Sampling with replacement generates new synthetic datasets of the same size, simulating what would happen if we collected a different 990 patients. The spread of metric values across resamples reflects the real uncertainty in the metric caused by who is in our dataset.

**Key caveat:** Bootstrap CIs measure **sampling variance** (uncertainty from dataset size), not model training variance. If the model itself is unstable (e.g., FT training with different seeds), you'd need to run CV multiple times with different seeds to capture that. For LP (only a linear head trained), model variance is small — the main uncertainty is the test set size.


In [ ]:
# ── Load pooled OOF predictions ───────────────────────────────────────────────
oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')
oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy')

# float32 softmax rows sum to ~0.9999 due to rounding; sklearn's roc_auc_score
# requires exact 1.0 — cast to float64 and renormalize (preserves rank order).
oof_probs = oof_probs.astype(np.float64)
oof_probs = oof_probs / oof_probs.sum(axis=1, keepdims=True)

# ── Per-fold summary ──────────────────────────────────────────────────────────
with open(CV_OUTPUT / 'fold_results.json') as f:
    fold_results = json.load(f)

print('Per-fold OOF AUROC:')
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['oof_auroc']:.4f}  (best val AUROC during training: {r['best_auroc']:.4f})")
fold_aurocs = [r['oof_auroc'] for r in fold_results]
print(f"  Mean: {np.mean(fold_aurocs):.4f}  Std: {np.std(fold_aurocs):.4f}")

print()

# ── Pooled OOF metrics ────────────────────────────────────────────────────────
m_oof = compute_metrics(oof_labels, oof_probs)
print(f'Pooled OOF metrics ({len(oof_labels)} images, ~{len(oof_labels)//len(fold_results)} per fold):')
print(f'  AUROC               : {m_oof["auroc"]:.4f}')
print(f'  Accuracy            : {m_oof["accuracy"]:.4f}')
print(f'  Cohen κ (quadratic) : {m_oof["kappa"]:.4f}')
print(f'  Macro sensitivity   : {m_oof["macro_sensitivity"]:.4f}')
print(f'  Macro specificity   : {m_oof["macro_specificity"]:.4f}')
print()
print('Per-class OOF sensitivity:')
for i, c in enumerate(CLASSES):
    print(f'  {c}: {m_oof["sensitivity"][i]:.4f}')

# ── Bootstrap 95% CI ─────────────────────────────────────────────────────────
def bootstrap_ci(labels, probs, metric_fn, n_boot=1000, seed=42, ci=95):
    rng = np.random.RandomState(seed)
    n = len(labels)
    vals = []
    _first_err = None
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        try:
            vals.append(metric_fn(labels[idx], probs[idx]))
        except Exception as e:
            if _first_err is None:
                _first_err = e
    if len(vals) == 0:
        raise RuntimeError(
            f"All {n_boot} bootstrap resamples failed.\n"
            f"First error: {type(_first_err).__name__}: {_first_err}"
        ) from _first_err
    if len(vals) < n_boot * 0.5:
        print(f"  Warning: only {len(vals)}/{n_boot} resamples succeeded. "
              f"First error: {type(_first_err).__name__}: {_first_err}")
    lo = np.percentile(vals, (100 - ci) / 2)
    hi = np.percentile(vals, 100 - (100 - ci) / 2)
    return np.mean(vals), lo, hi

auroc_fn  = lambda l, p: roc_auc_score(l, p, multi_class='ovr', average='macro',
                                        labels=list(range(NUM_CLASSES)))
kappa_fn  = lambda l, p: cohen_kappa_score(l, p.argmax(1), weights='quadratic')
acc_fn    = lambda l, p: accuracy_score(l, p.argmax(1))
m_sens_fn = lambda l, p: np.nanmean([
    (cm := confusion_matrix(l, p.argmax(1), labels=list(range(NUM_CLASSES))))[i,i] /
    max(cm[i,:].sum(), 1) for i in range(NUM_CLASSES)
])

print()
print('Bootstrap 95% CI (1000 resamples):')
for name, fn in [('AUROC', auroc_fn), ('Kappa', kappa_fn), ('Accuracy', acc_fn), ('Macro sens.', m_sens_fn)]:
    mean, lo, hi = bootstrap_ci(oof_labels, oof_probs, fn)
    print(f'  {name:<14}: {mean:.4f}  [{lo:.4f}, {hi:.4f}]')

# Per-class sensitivity CI
print()
print('Per-class sensitivity 95% CI:')
for i, c in enumerate(CLASSES):
    fn = lambda l, p, i=i: confusion_matrix(l, p.argmax(1), labels=list(range(NUM_CLASSES)))[i,i] / \
         max(confusion_matrix(l, p.argmax(1), labels=list(range(NUM_CLASSES)))[i,:].sum(), 1)
    mean, lo, hi = bootstrap_ci(oof_labels, oof_probs, fn)
    print(f'  {c}: {mean:.4f}  [{lo:.4f}, {hi:.4f}]')

In [ ]:
# ── Average 5 fold models' test predictions ───────────────────────────────────
test_probs_list = []
for fold in range(N_FOLDS):
    p = np.load(CV_OUTPUT / f'fold_{fold}_test_probs.npy').astype(np.float64)
    test_probs_list.append(p)
    print(f'Fold {fold} test shape: {p.shape}')

test_labels_ref = np.load(CV_OUTPUT / f'fold_0_test_labels.npy')
ensemble_probs  = np.mean(test_probs_list, axis=0)
ensemble_probs  = ensemble_probs / ensemble_probs.sum(axis=1, keepdims=True)

m_ens = compute_metrics(test_labels_ref, ensemble_probs)
print()
print(f'Test-set ENSEMBLE metrics (5-model average, {len(test_labels_ref)} images):')
print(f'  AUROC               : {m_ens["auroc"]:.4f}')
print(f'  Accuracy            : {m_ens["accuracy"]:.4f}')
print(f'  Cohen κ (quadratic) : {m_ens["kappa"]:.4f}')
print(f'  Macro sensitivity   : {m_ens["macro_sensitivity"]:.4f}')
print(f'  Macro specificity   : {m_ens["macro_specificity"]:.4f}')
print()
print('Per-class TEST sensitivity:')
for i, c in enumerate(CLASSES):
    print(f'  {c}: {m_ens["sensitivity"][i]:.4f}')
print()
print('Per-class TEST specificity:')
for i, c in enumerate(CLASSES):
    print(f'  {c}: {m_ens["specificity"][i]:.4f}')

# Bootstrap CI on test ensemble
print()
print('Test ensemble bootstrap 95% CI:')
for name, fn in [('AUROC', auroc_fn), ('Kappa', kappa_fn), ('Macro sens.', m_sens_fn)]:
    mean, lo, hi = bootstrap_ci(test_labels_ref, ensemble_probs, fn)
    print(f'  {name:<14}: {mean:.4f}  [{lo:.4f}, {hi:.4f}]')

np.save(CV_OUTPUT / 'test_ensemble_probs.npy',  ensemble_probs)
np.save(CV_OUTPUT / 'test_ensemble_labels.npy', test_labels_ref)

## Comparing CV results to the single-split runs

This cell builds a summary table with:
- **LP Run 1 / Run 2**: the two single-split test-set evaluations (hardcoded from your saved CSVs)
- **CV OOF**: pooled out-of-fold predictions on 990 patients (most reliable estimate)
- **CV Test ensemble**: 5-model average on the fixed 175-patient test set

**What to look for:**
- Does the CV OOF AUROC agree with the single-split test AUROCs? If yes, the test set was representative.
- Do the CI bounds overlap with the single-split point estimates? Differences within CI bounds are noise; differences outside them suggest real distributional shifts.
- Per-class sensitivity CIs tell you whether the R2/R3A numbers are trustworthy or just noise from tiny denominators.


In [ ]:
# ── Load single-split LP results from the saved CSVs ─────────────────────────
import warnings
lp_test_csv = 'output_dir/retfound_dinov2_modelA_lp/retfound_dinov2_modelA_lp/metrics_test.csv'
try:
    lp_test = pd.read_csv(lp_test_csv).iloc[0]
    lp_auroc = lp_test['roc_auc']
    lp_kappa = lp_test['kappa']
    lp_acc   = lp_test['accuracy']
    lp_msens = lp_test['macro_sensitivity']
    lp_mspec = lp_test['macro_specificity']
    lp_sens  = [lp_test[f'sensitivity_{i}'] for i in range(NUM_CLASSES)]
    print(f'Loaded LP test metrics: AUROC={lp_auroc:.4f}')
except Exception as e:
    warnings.warn(f'Could not load LP test CSV: {e}')
    lp_auroc = lp_kappa = lp_acc = lp_msens = lp_mspec = float('nan')
    lp_sens  = [float('nan')] * NUM_CLASSES

# ── Re-compute OOF and ensemble metrics (normalize to float64 for sklearn) ─────
oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')
oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
oof_probs  = oof_probs / oof_probs.sum(axis=1, keepdims=True)
m_oof      = compute_metrics(oof_labels, oof_probs)

ens_probs_f64 = np.load(CV_OUTPUT / 'test_ensemble_probs.npy')   # already float64 from cell above
ens_labels    = np.load(CV_OUTPUT / 'test_ensemble_labels.npy')

m_ens_l, m_ens_lo, m_ens_hi = bootstrap_ci(ens_labels, ens_probs_f64, auroc_fn)
_, oof_lo, oof_hi = bootstrap_ci(oof_labels, oof_probs, auroc_fn)

# ── Print comparison table ────────────────────────────────────────────────────
print('=' * 72)
print('MODEL A LP — SUMMARY COMPARISON')
print('=' * 72)
print(f'{"Metric":<24} {"LP (latest)":<16} {"CV OOF (990 pts)":<22} {"CV Test Ens.":<16}')
print('-' * 72)
print(f'{"AUROC":<24} {lp_auroc:<16.4f} {m_oof["auroc"]:.4f} [{oof_lo:.4f},{oof_hi:.4f}]  {m_ens["auroc"]:<.4f}')
print(f'{"Accuracy":<24} {lp_acc:<16.4f} {m_oof["accuracy"]:.4f}                    {m_ens["accuracy"]:.4f}')
print(f'{"Cohen κ":<24} {lp_kappa:<16.4f} {m_oof["kappa"]:.4f}                    {m_ens["kappa"]:.4f}')
print(f'{"Macro sensitivity":<24} {lp_msens:<16.4f} {m_oof["macro_sensitivity"]:.4f}                    {m_ens["macro_sensitivity"]:.4f}')
print(f'{"Macro specificity":<24} {lp_mspec:<16.4f} {m_oof["macro_specificity"]:.4f}                    {m_ens["macro_specificity"]:.4f}')
print()
print('Per-class sensitivity:')
print(f'{"Class":<8} {"LP":<10} {"CV OOF":<10} {"CV Ens.":<10}')
print('-' * 40)
for i, c in enumerate(CLASSES):
    print(f'{c:<8} {lp_sens[i]:<10.4f} {m_oof["sensitivity"][i]:<10.4f} {m_ens["sensitivity"][i]:<10.4f}')

print()
print('Key insight:')
print('  CV OOF AUROC is the most reliable estimate (990 patients, 5 folds).')
print('  Single-split differences within the CI bounds are noise, not signal.')
print('  If R2/R3A sensitivity CI bounds are wide, Phase 2 improvements targeting')
print('  those classes need at least a 2× CI width improvement to be meaningful.')

# ── Save summary ──────────────────────────────────────────────────────────────
summary = {
    'lp_test':      {'auroc': lp_auroc, 'kappa': lp_kappa, 'accuracy': lp_acc,
                     'macro_sensitivity': lp_msens, 'macro_specificity': lp_mspec,
                     'sensitivity': lp_sens},
    'cv_oof':       {'auroc': m_oof['auroc'], 'kappa': m_oof['kappa'],
                     'accuracy': m_oof['accuracy'],
                     'macro_sensitivity': m_oof['macro_sensitivity'],
                     'macro_specificity': m_oof['macro_specificity'],
                     'sensitivity': m_oof['sensitivity'].tolist(),
                     'auroc_ci_95': [float(oof_lo), float(oof_hi)]},
    'cv_test_ens':  {'auroc': m_ens['auroc'], 'kappa': m_ens['kappa'],
                     'accuracy': m_ens['accuracy'],
                     'macro_sensitivity': m_ens['macro_sensitivity'],
                     'macro_specificity': m_ens['macro_specificity'],
                     'sensitivity': m_ens['sensitivity'].tolist(),
                     'auroc_ci_95': [float(m_ens_lo), float(m_ens_hi)]},
}
with open(CV_OUTPUT / 'cv_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\nSummary saved to {CV_OUTPUT}/cv_summary.json')